## Setup

### Import Libraries

In [1]:
import pandas as pd
from matplotlib import pyplot as plt
import random
import datetime as dt
import numpy as np
import statistics as stats
import pickle
import json
import os
import copy
import inspect

# from configdb import configdb
# import duration_utils as du

# from rise_set.angle import Angle
# from rise_set.rates import ProperMotion
# from rise_set.visibility import Visibility
# from rise_set.astrometry import calculate_airmass_at_times, make_ra_dec_target
# from time_intervals.intervals import Intervals

# from models import ICRSTarget, OrbitalElementsTarget, SatelliteTarget

from pandas import notna

### Plotting Functions

In [2]:
def scatter(x_values, y_values, title, xlabel, ylabel, filename):
    plt.scatter(x_values, y_values, marker=".", color="black")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.savefig(os.path.join("plots", filename+".png"))
    plt.show()

In [3]:
def histogram(x_values, bin_edges, title, xlabel, ylabel, filename):
    plot = plt.hist(x_values, bin_edges)
    plt.xscale("log")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()

In [4]:
def flat_hist(x_values, bin_edges, bin_names, title, xlabel, ylabel, filename=None):
    data = np.histogram(x_values, bin_edges)
    plot = plt.bar(bin_names, data[0])
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()

In [5]:
def split_on_limits(df, column_name, limits):
    counts = []
    for i in range(len(limits)):
        bottom_limit = limits[i]
        if (i+1) < len(limits):
            top_limit = limits[i+1]
        else:
            top_limit = float("inf")
        counts.append(df[(df[column_name]>bottom_limit) & (df[column_name]<=top_limit)].shape[0])

        print(f"{bottom_limit} -> {top_limit}: {counts[i]}")

    labels = ["<="+str(x) for x in limits[1:]]
    labels.append(str(limits[-1])+"+")
    plt.bar(labels, counts)
    plt.show()

## Prep Data

### Load Request States

In [6]:
raw_data_folder = "/mnt/c/Users/ecf3/Documents/LCO/lco_data/extracted_data"
request_state = pickle.load(open(os.path.join(raw_data_folder, "requestgroups_request.pkl"), "rb"))[["id", "state"]]
request_state

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/c/Users/ecf3/Documents/LCO/lco_data/extracted_data\\requestgroups_request.pkl'

### Load Configuration Data

In [ ]:
def load_config_data():
    raw_data_folder = "/mnt/c/Users/ecf3/Documents/LCO/lco_data/extracted_data"
    config_data_list = []
    for i in range(2):
        temp = pickle.load(open(os.path.join(raw_data_folder, f"requestgroups_configuration_{i}.pkl"), "rb"))
        config_data_list.append(temp)
    config_data = pd.concat(config_data_list)
    return config_data

def group_config_data(config_data):
    num_request_ids = len(config_data["request_id"].unique())
    count = 0
    request_types = {}
    for request_id, group in config_data.groupby("request_id"):
        count += 1
        if count % 10000 == 0:
            print(f"\r{count} / {num_request_ids}", end="")
        request_types[request_id] = sorted(group["type"].unique().tolist())
    config_df = pd.Series(request_types).rename("exposure_types").to_frame()    
    return config_df

In [ ]:
condensed_config_data_filepath = "_condensed_config_data.pkldf"
if not os.path.isfile(condensed_config_data_filepath):
    config_df = group_config_data(load_config_data())
    config_df.to_pickle(condensed_config_data_filepath)
else:
    config_df = pd.read_pickle(condensed_config_data_filepath)

config_df

### Load Request and Proposal Data

In [ ]:
request_data = pd.read_pickle("condensed_2m/2m_large_all.pkldf")
proposal_data = pickle.load(open("proposals_proposal.pkl", "rb")).set_index("id")
invalid_requests = request_data.loc[request_data["windows"].apply(bool)]
valid_requests = request_data.loc[request_data["proposal_id"]!="NSF2022A-009", :]
proposals_2m = proposal_data.loc[proposal_data.index.isin(valid_requests["proposal_id"].unique()), :]
print("Original Number of Proposals:", len(proposal_data))
print("Number of Proposals used on the 2m telescopes:", len(proposals_2m))
print("Original Number of Requests:", len(request_data))
print("Requests with invalid windows:", len(invalid_requests))
print("Number of Valid Requests:", len(valid_requests))

### Calculate Availability Window Length

In [ ]:
def find_availability_length(og_windows):
    og_windows.sort(key=lambda x: x["start"])
    start = og_windows[0]["start"]
    end = og_windows[-1]["end"]
    td = end - start
    return (td.days*24*60*60 + td.seconds) / (60*60)

valid_requests["og_length"] = valid_requests["original_windows"].apply(find_availability_length)

### Get availability window start

In [ ]:
valid_requests["og_window_start"] = valid_requests["original_windows"].apply(lambda x: x[0]["start"])

### Get Target Types

In [ ]:
def get_target_type(target_constraint_pairs):
    target_types = set()
    for pair in target_constraint_pairs:
        target_types.add(pair["target"]["type"])
    if len(target_types) == 1:
        return target_types.pop()
    else:
        return "BOTH"
valid_requests["target_types"] = valid_requests["target_constraint_pairs"].apply(get_target_type)

### Merge Exposure Types

In [ ]:
valid_requests = valid_requests.merge(config_df, left_index=True, right_index=True)

### Merge Request State

In [ ]:
valid_requests = valid_requests.merge(request_state.set_index("id"), left_index=True, right_index=True)

### Calculate Flexibility

In [ ]:
def calculate_flexibility(row):
    total_observable_time = 0
    windows = row["windows"]
    for telescope, obs_intervals in windows.items():
        for i in range(0, len(obs_intervals.timepoints), 2):
            start = obs_intervals.timepoints[i]["time"]
            end = obs_intervals.timepoints[i+1]["time"]
            interval_length = end - start
            interval_length_seconds = interval_length.days*24*60*60 + interval_length.seconds
            total_observable_time += interval_length_seconds
    duration = row["total_duration"]
    return total_observable_time / duration

def calculate_theoretical_flexibility(row):
    theoretical_observable_time = 0
    windows = row["original_windows"]
    for i in range(len(windows)):
        start = windows[i]["start"]
        end = windows[i]["end"]
        interval_length = end - start
        interval_length_seconds = interval_length.days*24*60*60 + interval_length.seconds
        theoretical_observable_time += interval_length_seconds
    duration = row["total_duration"]
    return theoretical_observable_time / duration

valid_requests["flexibility"] = valid_requests.apply(calculate_flexibility, axis=1)
valid_requests["theoretical_flexibility"] = valid_requests.apply(calculate_theoretical_flexibility, axis=1)

### Split off Calibration Requests

Unsure about "MuSCAT Commissioning", it is not listed as a 'non-science' proposal.

In [ ]:
calibration_proposal_ids = proposals_2m[proposals_2m["non_science"]].index.tolist()
calibration_proposal_ids.append("MuSCAT Comissioning")
calibration_proposal_ids

In [ ]:
calibration = valid_requests[valid_requests["proposal_id"].isin(calibration_proposal_ids)]
requests = valid_requests[~valid_requests["proposal_id"].isin(calibration_proposal_ids)]
print("Calibration Requests:", len(calibration))
print("Science Requests:", len(requests))

In [ ]:
plt.bar(["Science", "Non-Science"], [len(requests), len(calibration)])
plt.title("Number of Science vs Non-Science Requests (for 2-meter Telescopes)")
plt.xlabel("Request Type")
plt.ylabel("# of 2m Requests")
plt.show()

In [ ]:
requests

In [ ]:
calibration

## Request Duration

### Plot Duration Distribution Quantiles

In [ ]:
quantiles = []
steps = []
step = 100
for i in range(int(step)+1):
    q = requests["total_duration"].quantile(i/step)
    print(i/step, q/60)
    quantiles.append(q/60)
    steps.append(i/step)

In [ ]:
dur_mean = requests["total_duration"].mean()
dur_std = requests["total_duration"].std()
outlier_limit = dur_mean/60 + 3*dur_std/60
outlier_limit

In [ ]:
requests[requests["total_duration"] > outlier_limit*60].sort_values("total_duration")

In [ ]:
scatter(steps, quantiles,
        "2m Request Duration Distribution",
        "Quantile",
        "Request Total Duration (mins)",
        "2m_duration_distribution")

scatter(steps[:-2], quantiles[:-2],
        "2m Request Duration Distribution (without outliers)",
        "Quantile",
        "Request Total Duration (mins)",
        "2m_duration_distribution_without_outliers")

### Suggest Values to split the durations
Splitting into Small, Medium, and Large distributions

Using boundaries of 600 and 1800 gives us a 35%, 38%, 27% split.

In [ ]:
split_on_limits(requests, "total_duration", [0, 600, 1800])

In [ ]:
split_on_limits(calibration, "total_duration", [0, 600, 1800])

### Split by Target Type

In [ ]:
target_types = requests["target_types"].value_counts()
print(target_types.index)
print(requests["target_types"].value_counts().to_list())

plt.bar([x for x in target_types.index], target_types.to_list())
plt.title("Log# of Requests by Target Type")
plt.yscale("log")
plt.ylabel("# of Requests")
plt.show()

plt.bar([x for x in target_types.index], target_types.to_list())
plt.ylabel("# of Requests")
plt.show()

### Duration vs Target Type

## Availability Windows

### Plot Availability Window Distribution

In [ ]:
quantiles = []
steps = []
step = 50
for i in range(int(step)+1):
    q = requests["og_length"].quantile(i/step)
    # print(i/step, q)
    quantiles.append(q)
    steps.append(i/step)

In [ ]:
scatter(steps, quantiles,
        "2m Availability Window Length Distribution",
        "Quantile (2% steps)",
        "Availability Window Length (Hrs)",
        "2m_availability_window_length_distribution")

In [ ]:
scatter(steps[:-1], quantiles[:-1],
        "2m Availability Window Length Distribution (without outliers)",
        "Quantile (2% steps)",
        "Availability Window Length (Hrs)",
        "2m_availability_window_length_distribution_without_outliers")

### Suggest values to split availability windows

Determining what values to use to set as the availability windows to test.

Using boundaries of 1hr and 24hrs gives us a 27%, 49%, 23% split.

In [ ]:
split_on_limits(requests, "og_length", [0, 1, 24, 72, 168])

In [ ]:
split_on_limits(calibration, "og_length", [0, 1, 24, 168])

In [ ]:
avw_limits = [0, 1, 24, 168]
avw_counts = []
for i in range(len(avw_limits)):
    bottom_limit = avw_limits[i]
    if (i+1) < len(avw_limits):
        top_limit = avw_limits[i+1]
    else:
        top_limit = 100000000
    avw_counts.append(requests[(requests["og_length"]>bottom_limit) & (requests["og_length"]<=top_limit)].shape[0])

    print(bottom_limit, top_limit, avw_counts[i])

labels = ["<="+str(x) for x in avw_limits[1:]]
labels.append(str(avw_limits[-1]) + "+")

plt.bar(labels, avw_counts)

In [ ]:
avw_limits = [0, 0.2, 0.5, 1, 6, 16, 24, 48, 72, 96, 120, 144, 168]
avw_counts = []
for i in range(len(avw_limits)):
    bottom_limit = avw_limits[i]
    if (i+1) < len(avw_limits):
        top_limit = avw_limits[i+1]
    else:
        top_limit = 100000000
    avw_counts.append(requests[(requests["og_length"]>bottom_limit) & (requests["og_length"]<=top_limit)].shape[0])

    print(bottom_limit, top_limit, avw_counts[i])

labels = [str(x) for x in avw_limits[1:]]
labels.append(str(avw_limits[-1]) + "+")

plt.bar(labels, avw_counts)

### Availability Windows with Target

In [ ]:
# ICRS Targets
vv1 = requests[requests["target_types"] == ('ICRS',)]

avw_limits = [0, 0.2, 0.5, 1, 6, 16, 24, 48, 72, 96, 120, 144, 168]
avw_counts = []
for i in range(len(avw_limits)):
    bottom_limit = avw_limits[i]
    if (i+1) < len(avw_limits):
        top_limit = avw_limits[i+1]
    else:
        top_limit = 100000000
    avw_counts.append(vv1[(vv1["og_length"]>bottom_limit) & (vv1["og_length"]<=top_limit)].shape[0])

    print(bottom_limit, top_limit, avw_counts[i])

labels = [str(x) for x in avw_limits[1:]]
labels.append(str(avw_limits[-1]) + "+")

plt.bar(labels, avw_counts)
plt.title("Availability Windows for ICRS Targets")

In [ ]:
vv1["og_length"].quantile(0.42)

In [ ]:
# Orbiting Element Targets
vv2 = requests[requests["target_types"] == ('ORBITAL_ELEMENTS',)]

avw_limits = [0, 0.2, 0.5, 1, 6, 16, 24, 48, 72, 96, 120, 144, 168]
avw_counts = []
for i in range(len(avw_limits)):
    bottom_limit = avw_limits[i]
    if (i+1) < len(avw_limits):
        top_limit = avw_limits[i+1]
    else:
        top_limit = 100000000
    avw_counts.append(vv2[(vv2["og_length"]>bottom_limit) & (vv2["og_length"]<=top_limit)].shape[0])

    print(bottom_limit, top_limit, avw_counts[i])

labels = [str(x) for x in avw_limits[1:]]
labels.append(str(avw_limits[-1]) + "+")

plt.bar(labels, avw_counts)
plt.title("Availability Windows for ORBITAL_ELEMENTS Targets")

In [ ]:
vv2["og_length"].quantile(0.54)

In [ ]:
weights = [4, 4, 4, 1]
for w in weights:
    print(w / sum(weights)*71380/6)

In [ ]:
for j in range(0, 11, 2):
    i = j/10
    print(f"{i}->{i+0.2}", requests[(requests["og_length"]>=i)&(requests["og_length"]<i+0.2)].shape)

In [ ]:
requests[requests["og_length"]<=1.0].value_counts("og_length").head(20)

### Availability Histogram

In [ ]:
max_val = requests["og_length"].max()
histogram(requests["og_length"],
          [0, 1.1, 24.1, 168.1, max_val],
          "Availability Window Histogram",
          "Availability Window Length (Hours)",
          "Number of Requests",
          "")

In [ ]:
requests[requests["og_length"]>200].sort_values("og_length")["og_length"]

In [ ]:
print(np.histogram(requests[requests["og_length"]>1.5]["og_length"], bins="auto"))
plt.hist(requests["og_length"], bins="auto")
# plt.yscale("log")
plt.xscale("log")
plt.show()

In [ ]:
eps = 10**(-5)
bins = [0, 0.3+eps, 1+eps, 3+eps, 10+eps, 30+eps, 100+eps, 300+eps, 1000+eps, 5000+eps]
x = requests["og_length"]
plt.hist(x, bins=bins)

# x = requests["og_length"]
# _, bins = np.histogram(x, bins="auto")
# logbins = np.logspace(np.log10(bins[0]),np.log10(bins[-1]),len(bins))
# plt.hist(x, bins=logbins)
plt.xscale("log")
# plt.yscale("log")
plt.ylim(1, 35000)

In [ ]:
max_value = round(requests["og_length"].max())
flat_hist(requests["og_length"],
          [0, 1.01, 24.01, 168.01, 10000000],
          ["<=1", "<=24", "<=168", f"168-{max_value}"],
          "Availability Window Histogram",
          "Availability Window Length (Hours)",
          "Number of Requests",
          "")

In [ ]:
from dateutil.relativedelta import relativedelta

In [ ]:
short = requests[requests["og_length"]<=1.0]

start_dates = []
month_start = dt.datetime(2020, 8, 1)
while month_start < dt.datetime(2022, 8, 2):
    start_dates.append(month_start)
    month_start += relativedelta(months=1)

counts = []

for i in range(len(start_dates)-1):
    start = start_dates[i]
    end = start_dates[i+1]
    month_requests = short[(short["og_window_start"]>start) & (short["og_window_start"]<end)]
    counts.append(len(month_requests))
    print(start, len(month_requests))

plt.scatter([str(x) for x in start_dates[:-1]], counts)
plt.ylim(0, 1000)
plt.show()
    
    


# to_timestamp = np.vectorize(lambda x: x.timestamp())
# time_stamps = to_timestamp(dt_array)
# np.histogram(time_stamps)

## Proposal Priority

### Plot Proposal Priority Distribution

In [ ]:
quantiles = []
steps = []
step = 50
for i in range(int(step)+1):
    q = proposals_2m["tac_priority"].quantile(i/step)
    print(i/step, q)
    quantiles.append(q)
    steps.append(i/step)

In [ ]:
scatter(steps, quantiles,
        "2m Proposal Priority Distribution",
        "Quantile (2% steps)",
        "Proposal Priority",
        "2m_proposal_priority_distribution")

In [ ]:
scatter(steps[:-2], quantiles[:-2],
        "2m Proposal Priority Distribution (without outliers)",
        "Quantile (2% steps)",
        "Proposal Priority",
        "2m_proposal_priority_distribution_without_outliers")

### Suggest Values to split Priorities
Determining what priority values to use to divide the proposals into bands.

Using boundaries of 20 and 25 gives us a 36%, 39%, 24% split.

In [ ]:
proposals_2m.sort_values("tac_priority", ascending=False)[["tac_priority"]].head(10)

In [ ]:
req_proposals = proposals_2m.drop(index=["COJ_calib", "OGG_calib", 
                                         "LCOEngineering", "FLOYDS standards",
                                         "MuSCAT Commissioning"])

prop_short = req_proposals[req_proposals["tac_priority"] <= 20.0]
prop_medium = req_proposals[(20.0 < req_proposals["tac_priority"]) & (req_proposals["tac_priority"] <= 25.0)]
prop_long = req_proposals[25.0 < req_proposals["tac_priority"]]

counts = [len(prop_short), len(prop_medium), len(prop_long)]
for c in counts:
    print(c/sum(counts)*100)

In [ ]:
proposals_2m[proposals_2m["non_science"]].index.tolist()

## Flexibility

In [ ]:
plt.scatter(requests["flexibility"], requests["state"])

In [ ]:
requests.value_counts("state")

Not sure how to represent this. But it looks like more flexible observations ARE getting scheduled, at least on the 2m telescopes, while the less flexible observations are more likely to get cancelled. This is a scenario that makes sense if the telescope is undersubscribed, as there should be enough space to fit everything in eventually.

In [ ]:
plt.scatter(requests["flexibility"], requests["total_duration"])
plt.yscale("log")
plt.ylim([170, 40000])
plt.xscale("log")
plt.show()

In [ ]:
requests["total_duration"].max()

In [ ]:
request_data

In [ ]:
a1 = pd.read_pickle("condensed_2m/2m_large_all.pkldf")

In [ ]:
a1.columns